# LLM Inference Mechanics

> **Source notes:** `ch02-llm-inference-mechanics`

This notebook explores the performance characteristics of LLM inference:
- **KV Cache Speedup** — measure the impact of key-value caching on generation speed
- **Prefill vs Decode Latency** — understand first-token vs per-token costs
- **Batching Throughput** — observe how batch processing affects tokens/second

All experiments use **GPT-2** via HuggingFace Transformers (local, no API key needed).

## 0 · Environment Setup

Install required packages and import libraries.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "transformers", "torch", "matplotlib", "numpy", "-q"], check=True)
print("✓ Packages installed")

In [ ]:
import time
import torch
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM

print("✓ Libraries imported")
print(f"  PyTorch version: {torch.__version__}")
print(f"  Device: {'GPU' if torch.cuda.is_available() else 'CPU'}")

## 1 · KV Cache Speedup Benchmark

**Key-value (KV) caching** is a critical optimization in autoregressive generation. Without it, each new token requires recomputing attention for the entire sequence. With caching, we only compute attention for the new token and reuse previous keys/values.

**Your task:**
1. Load GPT-2 small
2. Generate 100 tokens WITHOUT KV caching (set `use_cache=False`)
3. Generate 100 tokens WITH KV caching (set `use_cache=True`)
4. Measure time per token for both approaches
5. Calculate the speedup ratio

**Expected behavior:** KV caching should provide 2-5x speedup for moderate sequence lengths.

In [ ]:
# TODO: Implement this cell
#
# Instructions:
# 1. Load GPT-2 tokenizer and model: AutoTokenizer.from_pretrained("gpt2"), AutoModelForCausalLM.from_pretrained("gpt2")
# 2. Create a prompt string: "The future of artificial intelligence is"
# 3. Tokenize the prompt: input_ids = tokenizer.encode(prompt, return_tensors="pt")
# 4. Benchmark WITHOUT cache:
#    - Start timer
#    - Generate 100 tokens with use_cache=False, max_new_tokens=100, do_sample=False
#    - End timer, calculate time_per_token_no_cache
# 5. Benchmark WITH cache:
#    - Start timer
#    - Generate 100 tokens with use_cache=True, max_new_tokens=100, do_sample=False
#    - End timer, calculate time_per_token_cache
# 6. Calculate speedup = time_no_cache / time_cache
# 7. Print results
#
# Expected variables: time_per_token_no_cache, time_per_token_cache, speedup

In [ ]:
# Visualization
fig, ax = plt.subplots(figsize=(10, 6))
methods = ['Without KV Cache', 'With KV Cache']
times = [time_per_token_no_cache * 1000, time_per_token_cache * 1000]  # Convert to ms
colors = ['#e74c3c', '#2ecc71']

bars = ax.bar(methods, times, color=colors, alpha=0.8, edgecolor='black')
ax.set_ylabel('Time per Token (ms)', fontsize=12, fontweight='bold')
ax.set_title('KV Cache Speedup Benchmark', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.2f}ms',
            ha='center', va='bottom', fontweight='bold')

# Add speedup annotation
ax.text(0.5, max(times) * 0.9, f'Speedup: {speedup:.2f}x',
        ha='center', fontsize=14, fontweight='bold',
        bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.5))

plt.tight_layout()
plt.show()

## 2 · Prefill vs Decode Latency

LLM inference has two distinct phases:
- **Prefill (encode)**: Process the entire prompt in one forward pass. This is the "first token latency".
- **Decode**: Generate tokens one at a time autoregressively.

Prefill is expensive (grows with prompt length), but decode latency per token is more stable.

**Your task:**
1. Generate text with varying prompt lengths: 100, 500, 1000, 2000 tokens
2. For each prompt length, measure:
   - Prefill time (time to generate the first token)
   - Average decode time per token (for next 50 tokens)
3. Plot both metrics vs prompt length

**Expected behavior:** Prefill time grows with prompt length, decode time stays relatively constant.

In [ ]:
# TODO: Implement this cell
#
# Instructions:
# 1. Define prompt_lengths = [100, 500, 1000, 2000]
# 2. Initialize lists: prefill_times = [], decode_times = []
# 3. For each prompt_length:
#    - Create a prompt with approximately that many tokens (repeat a base string)
#    - Tokenize: input_ids = tokenizer.encode(prompt, return_tensors="pt")
#    - Measure prefill time:
#      * Start timer
#      * Generate 1 token: model.generate(input_ids, max_new_tokens=1, use_cache=True)
#      * End timer, store prefill_time
#    - Measure decode time:
#      * Start timer
#      * Generate 50 more tokens from the same prompt
#      * End timer, calculate avg_decode_time = total_time / 50
#    - Append prefill_time to prefill_times
#    - Append avg_decode_time to decode_times
# 4. Convert times to milliseconds
#
# Expected variables: prompt_lengths, prefill_times (in ms), decode_times (in ms)

In [ ]:
# Visualization
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(prompt_lengths, prefill_times, marker='o', linewidth=2,
        markersize=8, label='Prefill Time (First Token)', color='#e74c3c')
ax.plot(prompt_lengths, decode_times, marker='s', linewidth=2,
        markersize=8, label='Decode Time (Per Token)', color='#3498db')

ax.set_xlabel('Prompt Length (tokens)', fontsize=12, fontweight='bold')
ax.set_ylabel('Latency (ms)', fontsize=12, fontweight='bold')
ax.set_title('Prefill vs Decode Latency', fontsize=14, fontweight='bold')
ax.legend(fontsize=11, loc='upper left')
ax.grid(True, alpha=0.3)

# Add annotations for the longest prompt
ax.annotate(f'{prefill_times[-1]:.1f}ms',
            xy=(prompt_lengths[-1], prefill_times[-1]),
            xytext=(10, 10), textcoords='offset points',
            fontweight='bold', color='#e74c3c')
ax.annotate(f'{decode_times[-1]:.1f}ms',
            xy=(prompt_lengths[-1], decode_times[-1]),
            xytext=(10, -15), textcoords='offset points',
            fontweight='bold', color='#3498db')

plt.tight_layout()
plt.show()

## 3 · Batching Throughput

**Batching** allows the model to process multiple requests in parallel, amortizing compute overhead. Throughput (tokens/second) typically increases with batch size, though individual request latency may increase.

**Your task:**
1. Generate text with batch_size = 1, 4, 8, 16
2. For each batch size:
   - Create a batch of prompts (repeat the same prompt)
   - Generate 50 tokens per prompt
   - Measure total time
   - Calculate throughput = (batch_size × 50) / total_time
3. Plot throughput vs batch size

**Expected behavior:** Throughput increases with batch size (up to hardware limits), showing better GPU utilization.

**Note:** This exercise is most meaningful on GPU. On CPU, benefits may be limited.

In [ ]:
# TODO: Implement this cell
#
# Instructions:
# 1. Define batch_sizes = [1, 4, 8, 16]
# 2. Initialize list: throughputs = []
# 3. Define a base prompt string
# 4. For each batch_size:
#    - Create a batch of prompts: prompts = [base_prompt] * batch_size
#    - Tokenize: inputs = tokenizer(prompts, return_tensors="pt", padding=True)
#    - Start timer
#    - Generate 50 tokens: model.generate(**inputs, max_new_tokens=50, use_cache=True, pad_token_id=tokenizer.eos_token_id)
#    - End timer
#    - Calculate throughput = (batch_size * 50) / total_time
#    - Append to throughputs list
# 5. Print results for each batch size
#
# Expected variables: batch_sizes, throughputs (tokens per second)

In [ ]:
# Visualization
fig, ax = plt.subplots(figsize=(10, 6))

bars = ax.bar(range(len(batch_sizes)), throughputs,
              color=['#3498db', '#2ecc71', '#f39c12', '#e74c3c'],
              alpha=0.8, edgecolor='black')

ax.set_xlabel('Batch Size', fontsize=12, fontweight='bold')
ax.set_ylabel('Throughput (tokens/sec)', fontsize=12, fontweight='bold')
ax.set_title('Batching Throughput Analysis', fontsize=14, fontweight='bold')
ax.set_xticks(range(len(batch_sizes)))
ax.set_xticklabels(batch_sizes)
ax.grid(axis='y', alpha=0.3)

# Add value labels on bars
for i, bar in enumerate(bars):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.1f}',
            ha='center', va='bottom', fontweight='bold')

# Add efficiency gain annotation
if len(throughputs) > 1:
    gain = throughputs[-1] / throughputs[0]
    ax.text(0.5, max(throughputs) * 0.85,
            f'Batch-{batch_sizes[-1]} is {gain:.2f}x faster than Batch-{batch_sizes[0]}',
            ha='center', fontsize=12, fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.5),
            transform=ax.transAxes)

plt.tight_layout()
plt.show()

## Summary

Key takeaways from this notebook:

| Optimization | Impact | When to use |
|---|---|---|
| **KV Cache** | 2-5x speedup | Always enabled (default in production) |
| **Prefill optimization** | Reduces first-token latency | Critical for interactive apps, long prompts |
| **Batching** | Increases throughput | High-load inference servers, batch processing |

**Production implications:**
- KV cache is essential — without it, generation becomes quadratically expensive
- Prefill dominates latency for long prompts (>1000 tokens) — consider prompt compression or retrieval
- Batching maximizes GPU utilization but increases per-request latency — tune based on SLA requirements

**Next:** Explore quantization and model compression techniques for further efficiency gains.